# Do the Fisher-selected passbands improve the inference?

Two instruments that differ only in their passband edges, each reconstructing eight catalog truths with MAP + Laplace under the `electronic` noise model. `pedestal` is left out: the two-power family cannot represent its shape.

- **W7-X bands** (`band_scale_rule = "none"`): the published five bands, `s = 1` on every volume.
- **Fisher bands** (`band_scale_rule = "fisher_information"`, the design point): each volume's bands scaled by the `s` that minimizes the Cramer-Rao variance of `ln Te` at the state the reference equilibrium predicts there.

The figures and the comparison tables read the noisy fits, with the same noise in both arms.

```
python run_study.py --arms w7x_bands fisher_bands --workers 5   # beside this notebook
```


In [ ]:
import pathlib

import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    thomson_scattering_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    inference_metrics_utils,
    inference_store_utils,
    profile_table_utils,
    pubfig,
    sweep_utils,
    synthetic_target_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_plasma_state_estimation as plots,
)

pubfig.apply_style()

# Every path is anchored to the package, never to the kernel's working directory.
PACKAGE_DIRECTORY = pathlib.Path(plots.__file__).parents[2]
NOTEBOOK_DIRECTORY = (
    PACKAGE_DIRECTORY
    / "results_plots_tables"
    / "results"
    / "plasma_state_estimation"
    / "ts_band_scale"
)
PLOTS_DIRECTORY = NOTEBOOK_DIRECTORY / "plots"
TABLES_DIRECTORY = NOTEBOOK_DIRECTORY / "tables"
DATA_DIRECTORY = PACKAGE_DIRECTORY.parents[1] / "data"
DATASET = "equilibrium_a_3k_spline_input"
PARQUET = DATA_DIRECTORY / f"{DATASET}.parquet"
SETTINGS = DATA_DIRECTORY / f"{DATASET}_settings.json"

ARMS = ("w7x_bands", "fisher_bands")
ARM_LABELS = {"w7x_bands": "W7-X bands", "fisher_bands": "Fisher bands"}
ARM_OF_RULE = {
    "none": "w7x_bands",
    "fisher_information": "fisher_bands",
}
PROFILES = ("electron_density", "electron_temperature", "pressure")
EXCLUDED_TRUTHS = ()
NOISE_MODEL = "electronic"
THESIS_TRUTHS = ("wide_high_beta_gp", "peaked_1_parametric")
"""The truths whose profile figures the thesis shows; the tables cover all six."""


def arm_of(record):
    """Which band arm a run belongs to, read off the design point it was run with."""
    return ARM_OF_RULE[record.design_point.band_scale_rule]


loaded = {
    label: entry
    for label, entry in (
        (
            label,
            inference_store_utils.load_inference(
                inference_store_utils.INFERENCE_STORE, identifier
            ),
        )
        for label, identifier in sweep_utils.current_ids(
            NOTEBOOK_DIRECTORY / "inference_ids.json"
        ).items()
    )
    if entry[0].truth_name not in EXCLUDED_TRUTHS
    and entry[0].design_point.thomson_noise_model == NOISE_MODEL
    and arm_of(entry[0]) in ARMS
}
noisy = {
    label: entry
    for label, entry in loaded.items()
    if entry[0].settings.add_measurement_noise
}
metrics = inference_metrics_utils.sweep_metrics_table(
    entry[0] for entry in noisy.values()
)
metrics["arm"] = [arm_of(noisy[label][0]) for label in metrics["study_label"]]
truths = sorted(metrics["truth_name"].unique())
print(
    f"{len(loaded)} runs under {NOISE_MODEL}, {len(noisy)} of them noisy, "
    f"over {len(truths)} truths"
)
for truth in truths:
    counts = {
        arm: int(
            (
                (metrics["truth_name"] == truth)
                & (metrics["arm"] == arm)
                & (metrics["name"] == PROFILES[0])
            ).sum()
        )
        for arm in ARMS
    }
    print(
        f"  {truth:16s} "
        + ", ".join(f"{ARM_LABELS[arm]} {n} noisy fit(s)" for arm, n in counts.items())
    )

## 1. The band scales the Fisher rule applies

Per scattering volume: its flux label, scattering angle, the temperature the reference equilibrium predicts there, and the Fisher scale `s`. A scale on a bound of the admissible interval is set by the bound, not by the expected state.


In [ ]:
reference = sweep_utils.reference_truth(PARQUET, SETTINGS)
placements = {
    arm: inference_store_utils.load_placement(
        inference_store_utils.PLACEMENT_STORE,
        next(
            entry[0].diagnostics_placement_id
            for entry in loaded.values()
            if arm_of(entry[0]) == arm
        ),
    )
    for arm in ARMS
}
geometry = placements["fisher_bands"].thomson_geometry
response = placements["fisher_bands"].polychromator_response
_, expected_temperature = synthetic_target_utils.expected_volume_state(
    geometry, reference
)
volume_scales = pd.DataFrame(
    {
        "volume": np.arange(np.asarray(geometry.volume_positions).shape[0]),
        "rho": np.round(
            synthetic_target_utils.thomson_volume_flux_labels(
                geometry.volume_positions, reference
            ),
            3,
        ),
        "theta_deg": np.round(
            np.degrees(
                np.asarray(thomson_scattering_utils.volume_scattering_angles(geometry))
            ),
            1,
        ),
        "expected_te_ev": np.round(np.asarray(expected_temperature), 0),
        "fisher_s": np.round(np.asarray(response.band_scales, dtype=float), 3),
    }
)
volume_scales.to_csv(TABLES_DIRECTORY / "ts_band_scale_volume_scales.csv", index=False)
low_bound, high_bound = thomson_scattering_utils.band_scale_bounds(response)
# Tested on the unrounded scales: the table rounds them to three decimals.
fisher_scales = np.asarray(response.band_scales, dtype=float)
on_bound = int(
    np.isclose(fisher_scales, low_bound).sum()
    + np.isclose(fisher_scales, high_bound).sum()
)
print(
    f"Fisher band scales over the {len(volume_scales)} volumes: "
    f"{volume_scales['fisher_s'].min():.3f} to "
    f"{volume_scales['fisher_s'].max():.3f}, admissible {low_bound:.3f} to "
    f"{high_bound:.3f}, {on_bound} on a bound"
)
volume_scales

## 2. What each arm recovers

Both arms for each truth in `THESIS_TRUTHS`, drawn as in `mcmc_benchmark`: MAP solid, Laplace 5 to 95% band dashed, truth in black. The arms read different Thomson signals, so no per-volume points are drawn.


In [ ]:
for truth in THESIS_TRUTHS:
    records = {}
    for arm in ARMS:
        matching = [
            entry[0]
            for entry in noisy.values()
            if entry[0].truth_name == truth and arm_of(entry[0]) == arm
        ]
        if matching:
            records[arm] = min(matching, key=lambda record: record.settings.seed)
    if len(records) < len(ARMS):
        print(f"{truth}: not every arm stored, skipped")
        continue
    figure = plots.plot_method_band_comparison(
        pd.concat(
            [
                profile_table_utils.profile_band_table(record, arm)
                for arm, record in records.items()
            ]
        ),
        tuple(records),
        labels=ARM_LABELS,
        title=truth,
    )
    pubfig.save(figure, str(PLOTS_DIRECTORY / f"ts_band_scale_profiles_{truth}"))
    print(f"ts_band_scale_profiles_{truth}")

## 3. Did the MAPs converge?

Per arm, over the noisy runs: the largest `|grad U|_inf` at the MAP and the fewest converged starts. A start has converged when it stopped before the iteration cap.


In [ ]:
for arm in ARMS:
    runs = [entry[0] for entry in noisy.values() if arm_of(entry[0]) == arm]
    if not runs:
        continue
    print(
        f"{ARM_LABELS[arm]:12s} {len(runs)} runs, largest |grad U|_inf "
        f"{max(run.map_gradient_inf_norm for run in runs):.1e}, "
        f"fewest converged starts {min(run.n_map_starts_converged for run in runs)}"
    )

## 4. The comparison tables

One table per profile, a row per truth and a column block per arm: the **width** (median over the radius of the 5 to 95% band half-width, % of the local value) and the **RMSE** of the posterior mean (% of the truth's peak), with `mean` and `standard_deviation` rows over the truths. The difference table is the Fisher mean minus the W7-X mean, in percentage points; a difference below the `standard_deviation` row is between profile shapes, not between arms.


In [ ]:
tables = {}
for profile in PROFILES:
    table = inference_metrics_utils.arm_comparison_table(metrics, profile, ARMS)
    tables[profile] = table
    table.round(3).to_csv(TABLES_DIRECTORY / f"ts_band_scale_{profile}.csv")
difference = pd.DataFrame(
    {
        profile: {
            value: table.loc["mean", ("fisher_bands", value)]
            - table.loc["mean", ("w7x_bands", value)]
            for value in ("uncertainty_percent", "error_percent")
        }
        for profile, table in tables.items()
    }
).T
difference.round(3).to_csv(TABLES_DIRECTORY / "ts_band_scale_fisher_minus_w7x.csv")
print("Fisher minus W7-X, mean over the truths, in percentage points:")
print(difference.round(3).to_string())
display(tables["electron_temperature"].round(3))
display(tables["electron_density"].round(3))